<a href="https://colab.research.google.com/github/bojj27/hey-larry-wakeword/blob/main/hey_larry_wake_word_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Hey Larry — Wake Word Training

**Instructions:** 1) Runtime → Change runtime type → T4 GPU. 2) Runtime → Run all. 3) Wait ~1–2 hours. 4) Files download automatically at the end (`hey_larry.tflite` + `hey_larry.onnx`).

Training generates synthetic speech via Piper TTS (English, libritts_r-medium voice), augments it with background noise and room impulse responses, and trains a small DNN classifier using the openWakeWord framework. No edits needed — just Run All.

## Training your own openWakeWord models


**Quick-start:** If you just want to train a basic custom model for openWakeWord!

Follow the instructions for Step 1 below. Each time you change the wake word, click the play icon to the left of the title to generate a sample and make sure it sounds correct. The first time it takes a few minutes but subsequent runs will be quick.

Once you're satisfied with the pronounciation, go to the "Runtime" dropdown menu in the upper left of the page, and select "run all". Keep the tab open but feel free to do something else. After ~1 hour, your custom model will be ready and will automatically be downloaded to your computer!

If you are a Home Assistant user with the openWakeWord add-on, follow the instructions [here](https://github.com/home-assistant/addons/blob/master/openwakeword/DOCS.md#custom-wake-word-models) to install and enable your custom model.

---

If you are interested in learning more about the custom model training process (and increasing the accuracy of your custom models), read through each step in this notebook and try experimenting with different training parameters. If you have any questions or problems, feel free to start a discussion at the openWakeWord [repo](https://github.com/dscripka/openWakeWord/discussions).

In [ ]:
# @title  { display-mode: "form" }
# @markdown # 1. Test Example Training Clip Generation
# @markdown Since openWakeWord models are trained on synthetic examples of your
# @markdown target wake word, it's a good idea to make sure that the examples
# @markdown sound correct. Type in your target wake word below, and run the
# @markdown cell to listen to it.
# @markdown
# @markdown Tips for pronunciation:
# @markdown - Spell out phonetically: "hey siri" --> "hey_seer_e"
# @markdown - Spell out numbers: "2" --> "two"
# @markdown - No punctuation except "?" and "!"

import os
import sys
from IPython.display import Audio

if not os.path.exists('./piper-sample-generator'):
    !git clone https://github.com/rhasspy/piper-sample-generator
    # Pin to commit before Mar-2026 repackaging that broke `import generate_samples`
    # Parent of 1a8c49bd29b3a132721086ee88f2253f788594a8 (repackage commit)
    !cd piper-sample-generator && git checkout c9d824c0e2cce8bdeb000c219dc9cbc84df086ea
    os.makedirs('piper-sample-generator/models', exist_ok=True)
    # English voice: libritts_r-medium
    !wget -q -O piper-sample-generator/models/en_US-libritts_r-medium.onnx 'https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/libritts_r/medium/en_US-libritts_r-medium.onnx'
    !wget -q -O piper-sample-generator/models/en_US-libritts_r-medium.onnx.json 'https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/libritts_r/medium/en_US-libritts_r-medium.onnx.json'
    !pip install -q piper-tts piper-phonemize-cross
    !pip install -q webrtcvad
    !pip install -q torch==2.5.0 torchvision==0.20.0 torchaudio==2.5.0

if 'piper-sample-generator/' not in sys.path:
    sys.path.append('piper-sample-generator/')

import generate_samples

target_word = 'Hey_Larreey'  # @param {type:"string"}

def text_to_speech(text):
    generate_samples.generate_samples_onnx(
        text=text,
        max_samples=1,
        length_scales=[1.1],
        noise_scales=[0.7], noise_scale_ws=[0.7],
        output_dir='./', batch_size=1, auto_reduce_batch_size=True,
        file_names=['test_generation.wav'],
        model='piper-sample-generator/models/en_US-libritts_r-medium.onnx'
    )

text_to_speech(target_word)
Audio('test_generation.wav', autoplay=True)


In [ ]:
# Patch torch_audiomentations for torchaudio>=2.1 (set_audio_backend was removed).
# Self-sufficient: install the pinned package first, then locate + patch it.
%pip install -q torch-audiomentations==0.11.0
import importlib, importlib.util, re
importlib.invalidate_caches()
spec = importlib.util.find_spec('torch_audiomentations.utils.io')
assert spec is not None, 'torch_audiomentations failed to install — rerun this cell'
io_path = spec.origin
s = open(io_path).read()
s2 = re.sub(r'torchaudio\.set_audio_backend\([^)]*\)', 'pass  # removed in torchaudio>=2.1', s)
open(io_path,'w').write(s2)
print('patched:', io_path, '| changed:', s != s2)


In [ ]:
# @title  { display-mode: "form" }
# @markdown # 3. Train the Model
# @markdown This cell is now FULLY SELF-SUFFICIENT: it installs anything missing,
# @markdown downloads the background/training data if absent, trains, converts,
# @markdown and STOPS WITH A RED ERROR if any stage failed (no more false
# @markdown "Training complete!"). Safe to re-run — every step is skipped if
# @markdown its output already exists.

import os, sys, subprocess, importlib, importlib.util, shutil

# ---------- 0. target word (in case the listen-test cell was skipped) ----------
try:
    target_word
except NameError:
    target_word = 'Hey_Larreey'

# ---------- 1. openWakeWord repo ----------
if not os.path.exists('openwakeword/examples/custom_model.yml'):
    subprocess.check_call(['git', 'clone', 'https://github.com/dscripka/openwakeword'])

# ---------- 2. Python training deps (idempotent, one pip resolve) ----------
def _missing(mod):
    importlib.invalidate_caches()
    return importlib.util.find_spec(mod) is None

_DEPS = [
    ('torchinfo',             'torchinfo==1.8.0'),
    ('torchmetrics',          'torchmetrics==1.2.0'),
    ('speechbrain',           'speechbrain==0.5.14'),
    ('audiomentations',       'audiomentations==0.33.0'),
    ('torch_audiomentations', 'torch-audiomentations==0.11.0'),
    ('acoustics',             'acoustics==0.2.6'),
    ('pronouncing',           'pronouncing==0.2.0'),
    ('mutagen',               'mutagen==1.47.0'),
    ('scipy',                 'scipy==1.13.1'),  # matched to numpy<2.1 ABI — dep-drag protection
    ('datasets',              'datasets==2.14.6'),
    ('onnx2tf',               'onnx2tf==2.4.0'),
    ('onnx',                  'onnx'),
    ('onnxruntime',           'onnxruntime'),
    ('onnx_graphsurgeon',     'onnx_graphsurgeon'),
    ('sng4onnx',              'sng4onnx'),
    ('simple_onnx_processing_tools', 'simple_onnx_processing_tools'),
]
def _abi_broken(mod):
    try:
        importlib.import_module('scipy.io.wavfile' if mod == 'scipy' else mod)
        return False
    except ValueError:  # numpy.dtype size changed = binary incompatibility
        return True
    except Exception:
        return False

_to_install = [spec for mod, spec in _DEPS if _missing(mod) or (mod == 'scipy' and _abi_broken(mod))]
if _to_install:
    print('Installing missing training deps:', ', '.join(_to_install))
    # numpy<2.1 pinned in the SAME resolve so nothing drags numpy back up
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                           *_to_install, 'numpy<2.1'])
    importlib.invalidate_caches()

# numpy guard AFTER installs (Colab default 2.0.2 passes; >=2.1 breaks numba)
import numpy as _np_check
from packaging.version import Version
assert Version(_np_check.__version__) < Version('2.1'), (
    f'numpy {_np_check.__version__} >= 2.1 — use Runtime > Restart session, then re-run ONLY this cell.')
print(f'numpy {_np_check.__version__} OK (< 2.1)')

# torch_audiomentations backend patch (in case the patch cell was skipped)
_spec = importlib.util.find_spec('torch_audiomentations.utils.io')
if _spec is not None:
    import re as _re
    _s = open(_spec.origin).read()
    _s2 = _re.sub(r'torchaudio\.set_audio_backend\([^)]*\)',
                  'pass  # removed in torchaudio>=2.1', _s)
    if _s2 != _s:
        open(_spec.origin, 'w').write(_s2)
        print('patched torch_audiomentations for torchaudio>=2.1')

# ---------- 3. Training data (each block skipped if already present) ----------
import numpy as np
try:
    import scipy.io.wavfile
except ValueError:
    raise SystemExit('scipy/numpy BINARY MISMATCH (a dependency replaced one of them). '
                     'FIX: Runtime > Restart session, then re-run ONLY this cell — the pinned '
                     'scipy/numpy pair is already installed and loads cleanly after restart.')
from pathlib import Path
import datasets as hf_datasets
from tqdm import tqdm

if not os.path.exists('./mit_rirs'):
    os.mkdir('./mit_rirs')
    print('Downloading MIT room impulse responses (~1-2 min)...')
    _rir = hf_datasets.load_dataset('davidscripka/MIT_environmental_impulse_responses',
                                    split='train', streaming=True)
    for _row in tqdm(_rir):
        _name = _row['audio']['path'].split('/')[-1]
        scipy.io.wavfile.write(os.path.join('./mit_rirs', _name), 16000,
                               (_row['audio']['array'] * 32767).astype(np.int16))

if not os.path.exists('./audioset_16k'):
    print('Downloading AudioSet background clips (~5 min)...')
    os.makedirs('audioset', exist_ok=True)
    subprocess.check_call(['wget', '-q', '-O', 'audioset/bal_train09.tar',
        'https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/bal_train09.tar'])
    subprocess.check_call(['tar', '-xf', 'bal_train09.tar'], cwd='audioset')
    os.mkdir('./audioset_16k')
    _aud = hf_datasets.Dataset.from_dict(
        {'audio': [str(i) for i in Path('audioset/audio').glob('**/*.flac')]})
    _aud = _aud.cast_column('audio', hf_datasets.Audio(sampling_rate=16000))
    for _row in tqdm(_aud):
        _name = _row['audio']['path'].split('/')[-1].replace('.flac', '.wav')
        scipy.io.wavfile.write(os.path.join('./audioset_16k', _name), 16000,
                               (_row['audio']['array'] * 32767).astype(np.int16))

if not os.path.exists('./fma'):
    print('Downloading FMA music background clips (~5 min)...')
    os.mkdir('./fma')
    try:
        _fma = hf_datasets.load_dataset('rudraml/fma', name='small', split='train', streaming=True)
        _fma = iter(_fma.cast_column('audio', hf_datasets.Audio(sampling_rate=16000)))
        for _i in tqdm(range(3600 // 30)):  # ~1 hour of music
            _row = next(_fma)
            _name = _row['audio']['path'].split('/')[-1].replace('.mp3', '.wav')
            scipy.io.wavfile.write(os.path.join('./fma', _name), 16000,
                                   (_row['audio']['array'] * 32767).astype(np.int16))
    except Exception as _e:
        print(f'FMA streaming failed ({_e}); using synthetic colored-noise fallback')
        import numpy as _np2; _rng = _np2.random.default_rng(42)
        for _i in range(120):
            _t = _np2.linspace(0, 30, 16000 * 30)
            _sig = (_np2.sin(2 * _np2.pi * 220 * _t) * 0.05 +
                    _rng.standard_normal(len(_t)) * 0.05) * 32767
            scipy.io.wavfile.write(f'./fma/synthetic_{_i:03d}.wav', 16000, _sig.astype(_np2.int16))
        print(f'Generated {_i+1} synthetic music-substitute clips in ./fma/')

for _fname in ('openwakeword_features_ACAV100M_2000_hrs_16bit.npy',
               'validation_set_features.npy'):
    if not os.path.exists(_fname):
        print(f'Downloading {_fname} (large — a few min)...')
        subprocess.check_call(['wget', '-q',
            f'https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/{_fname}'])

# ---------- 4. Config ----------
import yaml
config = yaml.load(open('openwakeword/examples/custom_model.yml', 'r').read(), yaml.Loader)

number_of_examples = 10100  # @param {type:"slider", min:100, max:50000, step:50}
number_of_training_steps = 10000  # @param {type:"slider", min:0, max:50000, step:100}
false_activation_penalty = 1500  # @param {type:"slider", min:100, max:5000, step:50}
config['target_phrase'] = [target_word]
config['model_name'] = config['target_phrase'][0].replace(' ', '_')
config['n_samples'] = number_of_examples
config['n_samples_val'] = max(500, number_of_examples // 10)
config['steps'] = number_of_training_steps
config['target_accuracy'] = 0.5
config['target_recall'] = 0.25
config['output_dir'] = './my_custom_model'
config['max_negative_weight'] = false_activation_penalty

config['rir_paths'] = ['./mit_rirs']
config['background_paths'] = ['./audioset_16k', './fma']
config['false_positive_validation_data_path'] = 'validation_set_features.npy'
config['feature_data_files'] = {'ACAV100M_sample': 'openwakeword_features_ACAV100M_2000_hrs_16bit.npy'}
config['piper_sample_generator_path'] = './piper-sample-generator'  # explicit — default already set in YAML

with open('my_model.yaml', 'w') as _file:
    yaml.dump(config, _file)

# ---------- 4b. Patch generate_samples (ONNX shim + 16 kHz resample) ----------
# train.py at c9d824c calls generate_samples() without model= arg; the original
# function at that commit requires a .pt checkpoint path.  Appending this shim
# overrides it to delegate to generate_samples_onnx (also in the same file)
# with the libritts_r ONNX model and resamples output 22050 Hz -> 16 kHz
# (openWakeWord requires exactly 16 kHz input clips).
_gs_path = 'piper-sample-generator/generate_samples.py'
if os.path.exists(_gs_path) and '_onnx_shim_applied' not in open(_gs_path).read():
    _shim = '''
# ---- ONNX shim (auto-applied by training cell) ----
_onnx_shim_applied = True
def generate_samples(text, output_dir=None, model=None, max_samples=None, file_names=None,
        length_scales=(0.75, 1, 1.25), noise_scales=(0.667,), noise_scale_ws=(0.8,),
        max_speakers=None, phoneme_input=False, **kwargs):
    """Shim: delegates to generate_samples_onnx with ONNX model + 16 kHz resample."""
    import os as _os
    if model is None:
        _here = _os.path.dirname(_os.path.abspath(__file__))
        model = _os.path.join(_here, "models", "en_US-libritts_r-medium.onnx")
    generate_samples_onnx(text=text, output_dir=output_dir, model=model,
        max_samples=max_samples, file_names=file_names, length_scales=length_scales,
        noise_scales=noise_scales, noise_scale_ws=noise_scale_ws,
        max_speakers=max_speakers, phoneme_input=phoneme_input,
        **{k: v for k, v in kwargs.items()
           if k not in ("batch_size", "auto_reduce_batch_size", "slerp_weights")})
    import torchaudio as _ta
    from pathlib import Path as _P
    for _f in _P(output_dir).glob("*.wav"):
        _w, _s = _ta.load(str(_f))
        if _s != 16000:
            _ta.save(str(_f), _ta.functional.resample(_w, _s, 16000), 16000)
# ---- end shim ----
'''
    with open(_gs_path, 'a') as _f:
        _f.write(_shim)
    print('Patched piper-sample-generator with ONNX shim + 16 kHz resampling')

# ---------- 5. Train (each stage FAILS LOUDLY on error) ----------
def _run_stage(flag, what):
    print(f'\n===== {what} =====')
    _r = subprocess.run([sys.executable, 'openwakeword/openwakeword/train.py',
                         '--training_config', 'my_model.yaml', flag])
    if _r.returncode != 0:
        raise SystemExit(f'STAGE FAILED: {what} (train.py {flag}, exit {_r.returncode}). '
                         f'Scroll up to the FIRST red error above — that is the real problem.')

_run_stage('--generate_clips', 'Generating synthetic clips')
_run_stage('--augment_clips',  'Augmenting clips with noise/RIR')
_run_stage('--train_model',    'Training the model')

# ---------- 6. Convert ONNX -> tflite ----------
onnx_model_path = f"my_custom_model/{config['model_name']}.onnx"
name1 = f"my_custom_model/{config['model_name']}_float32.tflite"
name2 = f"my_custom_model/{config['model_name']}.tflite"

if not os.path.exists(onnx_model_path):
    raise SystemExit(f'TRAINING DID NOT COMPLETE — {onnx_model_path} was never produced. '
                     'Scroll up to the first red error.')

_onnx2tf = shutil.which('onnx2tf')
if _onnx2tf is None:
    raise SystemExit('onnx2tf CLI not found even after install — re-run this cell once; '
                     'if it persists, Runtime > Restart session and re-run this cell.')
_r = subprocess.run([_onnx2tf, '-i', onnx_model_path, '-o', 'my_custom_model/',
                     '-kat', 'onnx____Flatten_0'])
if _r.returncode != 0 or not os.path.exists(name1):
    raise SystemExit('ONNX->tflite conversion FAILED — scroll up for the onnx2tf error.')
shutil.move(name1, name2)

# ---------- 7. HONEST completion check ----------
_missing_files = [p for p in (onnx_model_path, name2) if not os.path.exists(p)]
if _missing_files:
    raise SystemExit(f'TRAINING DID NOT COMPLETE — missing: {_missing_files}')
print(f'\nTraining complete! Model: {config["model_name"]}')
print(f'Files: {onnx_model_path} and {name2}')


In [ ]:
# Download trained model files to your computer
from google.colab import files
import os

onnx_path = f"my_custom_model/{config['model_name']}.onnx"
tflite_path = f"my_custom_model/{config['model_name']}.tflite"

if os.path.exists(onnx_path):
    files.download(onnx_path)
    print(f'Downloaded: {onnx_path}')
else:
    print(f'WARNING: {onnx_path} not found — check training output above')

if os.path.exists(tflite_path):
    files.download(tflite_path)
    print(f'Downloaded: {tflite_path}')
else:
    print(f'WARNING: {tflite_path} not found — check training output above')

print('Done! Upload the .tflite file to your Home Assistant openWakeWord add-on.')
